# GPU：复用 CPU 缓存训练
选择 L4/A100 GPU 运行时，依次执行。先完成 CPU notebook，再运行这里。
与 CPU 使用相同的数据目录选择、本地 `/content/studies_links` 软链接和 `/content/local_cache` 缓存路径。
原始数据路径、数据版本、滤波和采样参数必须与预热阶段一致。
`--require-cache` 禁止补算原始信号；缺失缓存会立即报错。
`--resume auto` 恢复同一目录的 last.ckpt；总轮数仍是 EPOCHS，不会额外再加 EPOCHS。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive


def run_live(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


drive.mount("/content/drive")
REPO_URL = "https://github.com/wdongdong0708-jpg/brain2qwertyv1Test.git"
CODE = Path("/content/code")
WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")
if (CODE / ".git").exists():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
os.chdir(CODE)
for folder in ("data", "archives", "runs"):
    (WORKSPACE / folder).mkdir(parents=True, exist_ok=True)
raw_candidates = [WORKSPACE / "data/SpanishBCBL", WORKSPACE / "data/SpanishBCBL_Full"]
raw = next((path for path in raw_candidates if path.is_dir()), None)
if raw is None:
    raise FileNotFoundError(f"SpanishBCBL data directory not found; checked: {raw_candidates}")
raw = raw.resolve()
STUDIES_LINKS = Path("/content/studies_links")
STUDIES_LINKS.mkdir(parents=True, exist_ok=True)
for name in ("Pinet2024Eeg", "Pinet2024Meg"):
    alias = STUDIES_LINKS / name
    if alias.is_symlink() and alias.resolve() != raw:
        alias.unlink()
    if not alias.is_symlink():
        if alias.exists():
            raise FileExistsError(f"Expected a local symlink, but found an existing directory/file: {alias}")
        alias.symlink_to(raw, target_is_directory=True)
os.environ["BRAIN2QWERTY_STUDIES"] = str(STUDIES_LINKS)
os.environ["BRAIN2QWERTY_CACHE"] = "/content/local_cache"
Path(os.environ["BRAIN2QWERTY_CACHE"]).mkdir(parents=True, exist_ok=True)
os.environ["BRAIN2QWERTY_RESULTS"] = str(WORKSPACE / "runs")
print(f"Raw data: {raw}; study links: {STUDIES_LINKS}; cache: /content/local_cache")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
run_live([sys.executable, "-u", "-m", "scripts.colab_setup"])
PYTHON = "/content/b2q-venv/bin/python"

run_live([PYTHON, "-c", "import torch, brain2qwerty_v1; print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')"])


In [ ]:
os.environ["BRAIN2QWERTY_CACHE"] = "/content/local_cache"
run_live([PYTHON, "-u", "-m", "scripts.cache_archive", "unpack",
          "--archive", str(WORKSPACE / "archives/features.tar"),
          "--cache-root", os.environ["BRAIN2QWERTY_CACHE"]])


In [ ]:
MODALITY = "eeg"   # eeg / meg
SUBJECT = "all"    # 002 for one EEG subject; S2 for one MEG subject
EPOCHS = 300       # finite total budget; resume must keep the original value
SEED = 33
run_live([PYTHON, "-u", "-m", "brain2qwerty_v1", "train",
          "--modality", MODALITY, "--subject", SUBJECT,
          "--seed", str(SEED), "--epochs", str(EPOCHS),
          "--devices", "1", "--require-cache", "--resume", "auto"])


Checkpoint、逐步日志、每轮状态均直接写入 Drive 的 runs/。同一实验再次执行会恢复断点；模型结构、batch、种子和总轮数需保持一致。新实验请指定不同的 `--run-dir`。
